In [2]:
import pandas as pd
import numpy as np

our_cat = pd.read_csv('our_categories.csv')
supp_prod = pd.read_csv('supplier_products.csv')
labelled = pd.read_csv('labelled_products.csv')
sales = pd.read_csv('sales.csv')

In [3]:
unique_raw_category = supp_prod.supplier_category.dropna().unique()
raw_to_norm = {cat: "".join(cat.lower().split()) for cat in unique_raw_category}
supp_prod['norm_cat'] = supp_prod['supplier_category'].map(raw_to_norm)


In [4]:
supp_with_labels = supp_prod.merge(labelled, on="supplier_sku", how="left")
# for later to know which mistake would cost the most(based on revenue)
supp_with_labels = supp_with_labels.merge(sales[["supplier_sku", "revenue"]], on="supplier_sku", how="left")

labels_per_cat = (supp_with_labels.dropna(subset=["our_category_id"])
                  .groupby("norm_cat")["our_category_id"]
                  .agg(n_labelled="size", n_different_ids="nunique"))

## Which situation is each group in?

* **A. no label** = only the product model can decide
* **B. labelled, no conflict** = every hand label points to the same `our_category_id`
* **C. labelled, conflict** = the team put products of the same supplier category into different categories of ours

In [5]:
cat_info = pd.DataFrame({"norm_cat": supp_prod["norm_cat"].unique()})
cat_info = cat_info.merge(labels_per_cat, on="norm_cat", how="left").fillna({"n_labelled": 0, "n_different_ids": 0})

cat_info["situation"] = np.select(
    [cat_info["n_labelled"] == 0, cat_info["n_different_ids"] == 1],
    ["no label", "labels agree"],
    default="labels conflict",
)

cat_info["situation"].value_counts()

,count
situation,
no label,1709
labels agree,851
labels conflict,74


## Case B: Simplest answer: use only the team's labels
**What:** for each category that has labelled products, take the `our_category_id` that appears most often.
**Why:** it needs no model, it uses information the team already gave us, and it shows how far labels alone get us (coverage), so we know what is left for the model.
il folosesc doar acum ca s aarat ca daca m as folosi doar de cazul pe care ei mi l au prezentat as fi acoperit doar 61% din rezultate


In [6]:
labels_only = (supp_with_labels.dropna(subset=["our_category_id"])
               .groupby("norm_cat")["our_category_id"]
               .agg(lambda ids: ids.value_counts().index[0]))      # most frequent id in the category

n_cats = supp_prod["norm_cat"].nunique()
print(f"labels alone answer {len(labels_only)} of {n_cats} categories; {n_cats - len(labels_only)} are still open")

labels alone answer 925 of 2634 categories; 1709 are still open


**Result:** labels cover only ~61% of the categories, and most of those rest on 1–2 labelled products. That is why we need a model that reads the *other* ~50,000 products.


 ## Product model: from a product name to one of our categories

**4a. Merge retail/catering twins into one "family".**

In [7]:
our_cat["family"] = our_cat["path"].str.replace(r", (?:catering|retail) packs.*$", "", regex=True)
our_cat["is_catering"] = our_cat["path"].str.contains(", catering packs")

family_names = sorted(our_cat["family"].unique())
family_code = {name: code for code, name in enumerate(family_names)}
code_of_id = our_cat.set_index("our_category_id")["family"].map(family_code)

label_family = code_of_id.reindex(supp_with_labels["our_category_id"]).fillna(-1).astype(int).to_numpy()

**4b. Turn product names into numbers (TF-IDF).**

*Word* TF-IDF counts words and bigrams ("ice cream")
*Character* TF-IDF does the same with 3-5 letter pieces.

**Why:** the supplier writes `STRAWBERRIES`, `Strawberry`, `THICK STRAWBERRY`. Character pieces ("straw", "berr") match all of them, so typos and plurals do not hurt.

**4c. Linear SVM + softmax.**

`LinearSVC` learns one score per family from the labelled products only; softmax turns the scores into probabilities that sum to 1 for each product.

In [8]:
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

product_text = supp_with_labels["product_name"].str.lower()
n_products = len(supp_with_labels)
n_families = len(family_names)

def product_probabilities(known):
    words = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
    chars = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True, min_df=2)

    X_train = hstack([words.fit_transform(product_text[known]), chars.fit_transform(product_text[known])]).tocsr()
    X_all   = hstack([words.transform(product_text),            chars.transform(product_text)]).tocsr()

    model  = LinearSVC(C=0.5).fit(X_train, label_family[known])
    scores = model.decision_function(X_all) * 6
    scores -= scores.max(axis=1, keepdims=True)
    probs  = np.exp(scores)
    probs /= probs.sum(axis=1, keepdims=True)

    probabilities = np.zeros((n_products, n_families))
    probabilities[:, model.classes_] = probs
    return probabilities

known_all = label_family >= 0
probabilities = product_probabilities(known_all)
probabilities.shape

(18832, 145)

##  Vote per category

for every `norm_cat` we add up the probabilities of its products.
The family with the largest sum wins.
`confidence` = winner's sum / total sum (1.0 = all products agree).
Products with a hand label count as *certain*: their row is replaced by a 1 on the labelled family.

In [9]:
norm_cat_of_product = supp_with_labels["norm_cat"].to_numpy()

def vote_per_category(probabilities, known):
    probs = probabilities.copy()
    probs[known] = 0
    probs[np.flatnonzero(known), label_family[known]] = 1

    family_scores = pd.DataFrame(probs).groupby(norm_cat_of_product).sum()
    best_family = family_scores.idxmax(axis=1)
    confidence  = family_scores.max(axis=1) / family_scores.sum(axis=1)
    return probs, best_family, confidence

probs, best_family, confidence = vote_per_category(probabilities, known_all)

## Retail or catering?

In [12]:
import re
import numpy as np
import pandas as pd

unit_in_g_or_ml = {"kg": 1000, "g": 1, "gr": 1, "ml": 1, "cl": 10, "l": 1000, "ltr": 1000, "litre": 1000, "litres": 1000}
pack_pattern = re.compile(r"^(?:(\d+)\s*x\s*)?([\d.]+)\s*(kg|gr|g|ml|cl|ltr|litres|litre|l)\b")

def pack_total(pack_size):
    if pd.isna(pack_size):
        return np.nan
    m = pack_pattern.match(str(pack_size).lower().replace(",", ".").strip())
    if m is None:
        return np.nan
    how_many = int(m.group(1) or 1)
    return how_many * float(m.group(2)) * unit_in_g_or_ml[m.group(3)]

pack_to_total = {p: pack_total(p) for p in supp_with_labels["pack_size"].dropna().unique()}
# We map using .get() to assign np.nan for missing/unmatched pack sizes
total = supp_with_labels["pack_size"].map(pack_to_total).to_numpy()
big_by_pack = np.where(np.isnan(total), np.nan, (total >= 5000).astype(float))

is_catering_of_id = our_cat.set_index("our_category_id")["is_catering"]
big_by_label = is_catering_of_id.reindex(supp_with_labels["our_category_id"]).astype(float).to_numpy()

normal_id_of   = our_cat[~our_cat["is_catering"]].set_index("family")["our_category_id"]
catering_id_of = our_cat[our_cat["is_catering"]].set_index("family")["our_category_id"]

In [23]:
def choose_category_id(probs, best_family, known):
    p_in_family = probs[np.arange(n_products), best_family.reindex(norm_cat_of_product).to_numpy()]

    big = np.where(known, big_by_label, big_by_pack)
    weight = pd.Series(p_in_family * ~np.isnan(big))
    big_share = ((weight * np.nan_to_num(big)).groupby(norm_cat_of_product).sum()
                 / weight.groupby(norm_cat_of_product).sum()).reindex(best_family.index).fillna(0)

    winner = best_family.map(lambda code: family_names[code])
    use_catering = winner.isin(catering_id_of.index) & (big_share > 0.5)

    # Inlocuirea curata fara FutureWarning:
    our_category_id = pd.Series(
        np.where(use_catering, winner.map(catering_id_of), winner.map(normal_id_of)),
        index=best_family.index,
        dtype=int
    )

    return our_category_id, big_share, p_in_family

## Writing mapping.csv

In [24]:
def build_mapping(hidden_skus=()):
    known = known_all & ~supp_with_labels["supplier_sku"].isin(hidden_skus).to_numpy()

    probabilities = product_probabilities(known)
    probs, best_family, confidence = vote_per_category(probabilities, known)
    our_category_id, big_share, p_in_family = choose_category_id(probs, best_family, known)

    revenue = supp_with_labels["revenue"].to_numpy()
    mapping = pd.DataFrame({
        "our_category_id": our_category_id,
        "confidence": confidence,
        "big_pack_share": big_share,
        "n_products": pd.Series(norm_cat_of_product).value_counts(),
        "revenue": pd.Series(revenue).groupby(norm_cat_of_product).sum(),
        "revenue_at_risk": pd.Series(revenue * (1 - p_in_family)).groupby(norm_cat_of_product).sum(),
    })
    return mapping

mapping = build_mapping()
mapping.head()

,our_category_id,confidence,big_pack_share,n_products,revenue,revenue_at_risk
alest,6845,0.582105,0.000693,5,193854.02,129732.105256
alestandcri,6845,0.336743,0.000000,3,11032.10,5012.451450
alestou,7789,0.379098,0.000000,5,21844.92,7409.260340
alestout,7789,0.368723,0.000124,9,62185.66,49007.695344
apls-pears,7245,0.876961,0.000000,22,114038.84,9736.112597


In [25]:
raw_table = pd.DataFrame({"supplier_category": unique_raw_category})
raw_table["norm_cat"] = raw_table["supplier_category"].map(raw_to_norm)
answer = raw_table.merge(mapping[["our_category_id"]], left_on="norm_cat", right_index=True)

answer[["supplier_category", "our_category_id"]].to_csv("mapping.csv", index=False)

## Test on labels the model has not seen

**What:** split the labelled products into 4 groups *by category* (`GroupKFold`). Each round hides **all** labels of 1/4 of the categories, rebuilds the whole mapping and checks the hidden products: is `mapping[their category] == their true label`?

**Why:** if we scored on labels we trained on, the result would be too optimistic. Hiding whole categories also imitates the 1,051 categories that have no label at all, which is the hard case.

**Ceiling:** the best any one-id-per-category mapping could score on these products. Misses below the ceiling are ours; the rest comes from categories where the supplier mixed products.

In [26]:
from sklearn.model_selection import GroupKFold

labelled_rows = supp_with_labels[supp_with_labels["our_category_id"].notna()]
tested = []
for _, test_positions in GroupKFold(4).split(labelled_rows, groups=labelled_rows["norm_cat"]):
    hidden = labelled_rows.iloc[test_positions]
    m = build_mapping(hidden_skus=set(hidden["supplier_sku"]))
    tested.append(hidden.assign(predicted=hidden["norm_cat"].map(m["our_category_id"]),
                                confidence=hidden["norm_cat"].map(m["confidence"])))
tested = pd.concat(tested)
tested["correct"] = tested["predicted"] == tested["our_category_id"]

best_possible = tested.groupby(["norm_cat", "our_category_id"]).size().groupby("norm_cat").max().sum() / len(tested)
print(f"hidden labelled products placed correctly: {tested['correct'].mean():.1%}")
print(f"best any one-id-per-category mapping could do: {best_possible:.1%}")

hidden labelled products placed correctly: 82.1%
best any one-id-per-category mapping could do: 94.1%


**Is the confidence honest?** Grouped the hidden test by `confidence`. For categories with at least 3 hidden labels we compare with the *majority* label of the category (a single odd product should not decide).

In [27]:
per_cat = tested.groupby("norm_cat").agg(n=("correct", "size"), predicted=("predicted", "first"),
                                          confidence=("confidence", "first"),
                                          majority_label=("our_category_id", lambda s: s.value_counts().index[0]))
per_cat = per_cat[per_cat["n"] >= 3]
per_cat["correct"] = per_cat["predicted"] == per_cat["majority_label"]
per_cat["confidence_bucket"] = pd.cut(per_cat["confidence"], [0, .5, .7, .85, .95, 1.0001])

print(f"categories with 3+ hidden labels: {len(per_cat)}, correct: {per_cat['correct'].mean():.1%}")
per_cat.groupby("confidence_bucket", observed=True)["correct"].agg(categories="size", correct="mean").round(3)

categories with 3+ hidden labels: 97, correct: 86.6%


,categories,correct
confidence_bucket,,
"(0.0, 0.5]",35,0.686
"(0.5, 0.7]",18,0.889
"(0.7, 0.85]",19,1.000
"(0.85, 0.95]",14,1.000
"(0.95, 1.0]",11,1.000


## Which mistakes would cost most, and where is it weakest?

**What:** `revenue_at_risk` of a category = revenue of the products that the model believes do **not** belong with the chosen category (revenue * (1 - probability it belongs)). We rank categories by it.

**Why:** a mistake in a small category is cheap; in a large mixed category it moves a lot of money. This is the list the team should review first. Low `confidence` marks categories where the supplier mixed products, so no single id can be right for all.

In [28]:
example_name = supp_prod.groupby("norm_cat")["supplier_category"].first()
path_of_id   = our_cat.set_index("our_category_id")["path"]

weak_spots = mapping.assign(
    supplier_category=example_name,
    chosen_category=mapping["our_category_id"].map(path_of_id),
    n_labelled=cat_info.set_index("norm_cat")["n_labelled"],
).sort_values("revenue_at_risk", ascending=False)
weak_spots.to_csv("weak_spots.csv")

weak_spots.head(15)[["supplier_category", "n_products", "revenue", "confidence", "revenue_at_risk", "chosen_category"]].round(2)

,supplier_category,n_products,revenue,confidence,revenue_at_risk,chosen_category
pcklscndmn,pcklscndmn,41,522616.59,0.37,389273.10,Food cupboard > Sauces > Pickles & condiments
citr.frt,Citr.frt,52,455735.79,0.36,386764.66,Fruit & veg > Fruit > Citrus fruit
soft-squashgrp3,Soft-Squash Grp 3,35,422010.09,0.50,372180.32,Drinks > Soft drinks > Squash
dips-deli,Dips - Deli,12,367114.48,0.48,316315.03,Chilled > Dips & sides > Dips
bkd.beansandtndfish,BKD.BEANS AND TND FISH,9,323401.46,0.44,312672.84,Food cupboard > Tins > Baked beans
chs>hrdchs,CHS > HRD CHS,16,315021.57,0.49,282040.51,Dairy & eggs > Cheese > Hard cheese
pts_pet.cr,PTS_PET.CR,11,363462.41,0.54,281188.83,Baby & pets > Pets > Pet care
tinnefrt,TINNEFRT,45,384792.77,0.62,274494.14,Food cupboard > Tins > Tinned fruit
nap-wipesandpetcare,NAP-WIPES AND PET CARE,11,281734.75,0.52,242818.22,Baby & pets > Baby > Nappies & wipes
garl.brd&rolbaps,GARL.BRD & ROL BAPS,14,287156.07,0.46,235686.66,Bakery > Bread > Garlic bread


In [29]:
total_revenue = mapping["revenue"].sum()
shaky = mapping["confidence"] < 0.7
mixed_packs = mapping["big_pack_share"].between(0.1, 0.9) & mapping["our_category_id"].isin(our_cat.loc[our_cat["is_catering"] |
                                                          our_cat["family"].isin(catering_id_of.index), "our_category_id"])

print(f"categories with confidence < 0.7 : {shaky.sum()} holding {mapping.loc[shaky, 'revenue'].sum() / total_revenue:.0%} of revenue")
print(f"categories mixing retail + catering packs: {mixed_packs.sum()}")
print(f"categories with no hand label: {(cat_info['situation'] == 'no label').sum()} (the model alone decides)")
print("our categories that receive no supplier category:")
print(our_cat.loc[~our_cat["our_category_id"].isin(mapping["our_category_id"]), "path"].to_string(index=False))

categories with confidence < 0.7 : 1045 holding 50% of revenue
categories mixing retail + catering packs: 29
categories with no hand label: 1709 (the model alone decides)
our categories that receive no supplier category:
        Drinks > Soft drinks > Lemonade & mixers
Food cupboard > Breakfast > Jams & sweet spreads
   Food cupboard > Stock & gravy > Stock & gravy
       Frozen > Frozen meals > Burgers & nuggets
                   Fruit & veg > Fruit > Bananas
                 Meat & fish > Fish > Fresh fish
